In [1]:
import pandas as pd
import numpy as np
import streamlit as st

In [2]:
def load_data(filepath: str) -> pd.DataFrame:
    try :
        if filepath[-3:] == 'csv':
            return pd.read_csv(filepath)
        elif filepath[-4:] == 'xlsx' or filepath[-3:] == 'xls':
            return pd.read_excel(filepath)
    except Exception as e:
        print(f"Error reading file {filepath}: {e}")
        return pd.DataFrame()


In [21]:
def group_and_aggregate_data(df: pd.DataFrame, group_by_column: str, agg_func) -> pd.DataFrame:
    try:
        agg_func = agg_func if isinstance(agg_func, str) else agg_func.__name__
        return df.drop(columns='ballot_code').groupby(group_by_column).aggregate(agg_func)
    except Exception as e:
        print(f"not a aggregate function: {e}")
        return pd.DataFrame()

In [44]:
df = group_and_aggregate_data(load_data("knesset_25.xlsx"), "city_name", "sum")
df

,party_avoda,party_shahar_kalkali_hadash,party_bayit_yehudi,party_agudat_israel,party_daled,party_vavmem,party_shahar_koach_hevrati,party_kama,party_koach_lehashpia,party_tzomet,...,party_tze'irim_bo'arim,party_manhigut_hevratit,party_kol_hasviva_vehachai,party_halev_hayehudi,party_seder_chadash,party_kol,party_beometz_bishvilech,party_kavod_umasoret,party_shas,party_daat_tov_vera
city_name,,,,,,,,,,,,,,,,,,,,,
אבו גווייעד שבט,1,0,0,0,4,38,0,0,1,0,...,1,0,0,0,0,0,0,0,4,3
אבו גוש,14,1,1,3,1263,312,0,0,0,0,...,2,7,1,0,1,1,3,0,4,0
אבו סנאן,34,0,3,0,677,2030,4,1,2,0,...,1,4,1,3,1,6,9,0,12,1
אבו עבדון שבט,0,0,0,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
אבו קורינאת שבט,5,0,1,0,10,65,0,0,1,1,...,0,1,0,0,2,1,0,0,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
תקומה,3,2,42,1,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,13,0
תקוע,25,18,266,13,0,1,0,0,0,0,...,1,1,1,3,0,0,26,0,27,0
תראבין אצאנע שבט,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [76]:
def remove_sparse_columns(df: pd.DataFrame, threshold: int) -> pd.DataFrame:
     return df[df.sum()[df.sum() > threshold].index]

In [91]:
result = remove_sparse_columns(df, threshold=1000000)
result

,party_likud
city_name,
אבו גווייעד שבט,12
אבו גוש,208
אבו סנאן,405
אבו עבדון שבט,0
אבו קורינאת שבט,9
...,...
תקומה,142
תקוע,353
תראבין אצאנע שבט,28


In [187]:
def dimensionality_reduction(df: pd.DataFrame, num_components: int, meta_columns: list[str]) -> pd.DataFrame:
    # save the metadata and save it for later use
    metadata = df[meta_columns]
    #print(metadata)

    # remove the metadat for the df
    metadata_removed = df.drop(columns=meta_columns).dropna()

    # standardize the data shift the data around zero and remove nones
    df_standardize = ((metadata_removed - metadata_removed.mean()) / metadata_removed.std())

    # center the data around 0 and find the covariance
    cov_matrix = np.cov(df_standardize.T)

    # find the eigenvalues and eigenvectors
    eig, eig_matrix = np.linalg.eig(cov_matrix)

    # find the sorted indexes from the large to small
    sorted_indexes = np.argsort(eig)[::-1]

    # sort by the index
    sorted_eig_matrix = eig_matrix[:, sorted_indexes]

    # select the top eigenvectors (columns) according to the argument num_components
    top_eig = sorted_eig_matrix[:, :num_components]

    # project the data into lower dimension plane according to the incorporate of the eigenvectors
    reduced_data = df_standardize.dot(top_eig)
    #print(reduced_data)

    # columns name
    #columns_names = [f"PC{i+1}" for i in range(num_components)]

    # rename the columns to pc1, pc2 ...
    #reduced_df = pd.DataFrame(reduced_data, columns=columns_names)

    # combine the metadata with the reduced dataframe
    final_df = pd.concat([metadata, reduced_data], axis=1).reset_index(drop=True)

    return final_df

In [188]:
dimensionality_reduction(df, 10, ["party_avoda", "party_shahar_kalkali_hadash", "party_ani_veata"])

,party_avoda,party_shahar_kalkali_hadash,party_ani_veata,0,1,2,3,4,5,6,7,8,9
0,1,0,0,-0.469711,-0.188915,-0.037500,-0.413765,0.244814,-0.119780,0.311355,-0.066876,-0.195196,0.283165
1,14,1,3,-0.269931,-2.096818,0.395062,0.374850,-0.191165,-0.185572,-0.171015,0.159006,-0.356472,-0.166790
2,34,0,2,2.358698,-3.234926,0.323469,-0.902711,0.271163,1.139275,-0.097546,0.008711,1.203732,0.675111
3,0,0,0,-0.973964,0.161524,-0.078367,-0.098957,-0.054927,-0.061085,-0.042043,-0.025733,-0.028016,-0.013480
4,5,0,0,-0.040868,-0.837427,-0.115353,-0.377150,0.149537,-0.328050,1.056889,-0.187888,-0.400684,0.668105
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1211,3,2,0,-0.929401,0.216650,-0.058271,-0.090199,-0.090588,-0.081793,-0.075258,0.016722,0.005204,-0.031087
1212,25,18,0,-0.118282,0.279944,0.063144,-0.315912,-0.453273,-0.412378,-0.037128,0.027657,-0.071952,-0.413978
1213,0,0,0,-0.972704,0.161277,-0.078190,-0.098527,-0.054761,-0.061305,-0.038569,-0.026969,-0.028442,-0.014737
1214,1,0,0,-0.960443,0.134120,-0.071364,-0.091909,-0.056777,-0.065071,0.003783,-0.042867,-0.036788,-0.029118
